In [0]:
%run /Configurations/Init_Scripts

In [0]:

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")


dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')
rawpath_flag=ExternalLocation_raw+'/MasterData/FlagData/FlagType.csv'

dbutils.widgets.text("ExternalLocationName_silver", "/mntprod_silver")
ExternalLocationName_silver = dbutils.widgets.get("ExternalLocationName_silver")
silverpath_flag=ExternalLocationName_silver+'/DIMFlagType'

In [0]:
spark.conf.set("spark.sql.catalog.current", CatalogName)

In [0]:
#Declaring Source and Destination Paths
# rawpath_flag = '/mnt/raw/MasterData/FlagData/FlagType.csv'

# silverpath_flag = '/mnt/silver/DIMFlagType'

schema = StructType([
    StructField("PromotionName", StringType(), True),
    StructField("FlagType", StringType(), True),
    StructField("FlagRank", StringType(), True),
    StructField("Active", IntegerType(), True),
    StructField("FlagSubType", StringType(), True)
])

In [0]:
df_flagtype = spark.read.options(header='true', delimiter=',').schema(schema).csv(rawpath_flag)

#Join with promotion table to derive PromotionUUID 
df_promotion = spark.read.table("promotion.dim_promotion")

df_result = (df_promotion.join(df_flagtype, df_promotion.PromotionName == df_flagtype.PromotionName, 'INNER')
                           .select(df_promotion.PromotionUUID,
                                   df_flagtype.FlagType,
                                   df_flagtype.FlagSubType,
                                   df_flagtype.FlagRank,
                                   df_flagtype.Active)
                           .withColumn('FlagTypeUUID',expr('uuid()'))
                           .withColumn('CreatedBy',lit('ADB_FlagType'))
                           .withColumn('CreatedDate',current_timestamp())
                           .withColumn('UpdatedBy',lit('ADB_FlagType'))
                           .withColumn('UpdatedDate',current_timestamp()))
df_result.display()

In [0]:
targetDF = DeltaTable.forPath(spark, silverpath_flag)  
(targetDF.alias("tgt")
  .merge(df_result.alias("src"), "src.PromotionUUID = tgt.PromotionUUID and src.FlagType = tgt.FlagType")
  .whenMatchedUpdate(
    set = {
        "tgt.FlagRank": "src.FlagRank",
        "tgt.FlagSubType": "src.FlagSubType",
        "tgt.UpdatedDate": "src.UpdatedDate",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.Active": "src.Active"
           }
  )
  .whenNotMatchedInsert(
    values = {
        "tgt.PromotionUUID": "src.PromotionUUID",
        "tgt.FlagTypeUUID":"src.FlagTypeUUID",
        "tgt.FlagType": "src.FlagType",
        "tgt.FlagSubType": "src.FlagSubType",
        "tgt.FlagRank": "src.FlagRank",
        "tgt.Active": "src.Active",
        "tgt.CreatedDate": "src.CreatedDate",
        "tgt.CreatedBy": "src.CreatedBy",
        "tgt.UpdatedDate": "src.UpdatedDate",
        "tgt.UpdatedBy": "src.UpdatedBy"
    }
  )
  .execute()
)